In [18]:
import pandas as pd

In [19]:
weather = pd.read_csv('../data/raw/nasa_monthly_weather_2014_2025.csv')
df_final = pd.read_csv('../data/processed/clean_crop_yield_data.csv')


In [20]:
# Extract Year and Month

weather["Year"] = weather["YearMonth"] // 100
weather["Month"] = weather["YearMonth"] % 100

print(weather.head())

                         State                  District  YearMonth  \
0  Andaman And Nicobar Islands  North And Middle Andaman     201401   
1  Andaman And Nicobar Islands  North And Middle Andaman     201402   
2  Andaman And Nicobar Islands  North And Middle Andaman     201403   
3  Andaman And Nicobar Islands  North And Middle Andaman     201404   
4  Andaman And Nicobar Islands  North And Middle Andaman     201405   

   Rainfall_mm  Temperature_C  Year  Month  
0         0.66          26.14  2014      1  
1         0.01          26.26  2014      2  
2         0.02          28.18  2014      3  
3         0.36          30.24  2014      4  
4         9.14          29.29  2014      5  


In [21]:
# --- MONSOON (Jun–Sep) ---
monsoon = (
    weather[weather["Month"].isin([6,7,8,9])]
    .groupby(["State","District","Year"])
    .agg({
        "Rainfall_mm":"sum",
        "Temperature_C":"mean"
    })
    .reset_index()
)

monsoon.columns = [
    "State","District","Year",
    "Monsoon_Rainfall",
    "Monsoon_Temp"
]


# --- RABI (Oct–Mar) ---
rabi = (
    weather[weather["Month"].isin([10,11,12,1,2,3])]
    .groupby(["State","District","Year"])
    .agg({
        "Rainfall_mm":"sum",
        "Temperature_C":"mean"
    })
    .reset_index()
)

rabi.columns = [
    "State","District","Year",
    "Rabi_Rainfall",
    "Rabi_Temp"
]


# --- SUMMER (Mar–May) ---
summer = (
    weather[weather["Month"].isin([3,4,5])]
    .groupby(["State","District","Year"])
    .agg({
        "Rainfall_mm":"sum",
        "Temperature_C":"mean"
    })
    .reset_index()
)

summer.columns = [
    "State","District","Year",
    "Summer_Rainfall",
    "Summer_Temp"
]

In [22]:
df_seasonal = df_final.merge(monsoon, on=["State","District","Year"], how="left")
df_seasonal = df_seasonal.merge(rabi, on=["State","District","Year"], how="left")
df_seasonal = df_seasonal.merge(summer, on=["State","District","Year"], how="left")

print("After seasonal merge:", df_seasonal.shape)
print(df_seasonal.isnull().sum())

After seasonal merge: (44324, 13)
State                 0
District              0
Crop                  0
Season                0
Area                  0
Yield                 0
Year                  0
Monsoon_Rainfall    961
Monsoon_Temp        961
Rabi_Rainfall       961
Rabi_Temp           961
Summer_Rainfall     961
Summer_Temp         961
dtype: int64


In [23]:
df_seasonal = df_seasonal.dropna().reset_index(drop=True)
print("After droping nulls:", df_seasonal.shape)
print(df_seasonal.isnull().sum())

After droping nulls: (43363, 13)
State               0
District            0
Crop                0
Season              0
Area                0
Yield               0
Year                0
Monsoon_Rainfall    0
Monsoon_Temp        0
Rabi_Rainfall       0
Rabi_Temp           0
Summer_Rainfall     0
Summer_Temp         0
dtype: int64


In [24]:
# Save cleaned dataset
df_seasonal.to_csv(
    "../data/processed/clean_yield_with_climate.csv",
    index=False
)

print("Saved successfully ✅")

Saved successfully ✅
